In [3]:
import os
import asyncio
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

from datasets import load_dataset

ds = load_dataset("jrosseruk/cocktails-with-instructions")

# Convert the 'train' split (or another available split) into a pandas DataFrame
df = ds["train"].to_pandas()

In [4]:
df.head()

,title,glass,garnish,ingredients,recipe,synthetic_instructions
0,Abacaxi Ricaço,Pineapple shell (frozen) glass,Cut a straw sized hole in the top of the pinea...,"[['1 whole', 'Pineapple (fresh)'], ['9 cl', 'H...",Cut the top off a small pineapple and carefull...,Chill the hollow pineapple shell in the freeze...
1,Abbey,Coupe glass,Orange zest twist,"[['4.5 cl', 'Rutte Dry Gin'], ['2.25 cl', 'Lil...",SHAKE all ingredients with ice and fine strain...,"Begin by gathering the Abbey, a bright, citrus..."
2,A.B.C. Cocktail,Nick & Nora glass,Lemon zest twist & Luxardo Maraschino cherry,"[['7 fresh', 'Mint leaves'], ['3 cl', 'Tawny p...",TEAR mint and place in shaker. Add other ingre...,Tear the fresh mint leaves and drop them into ...
3,Absinthe Cocktail,Coupe glass,Mint leaf,"[['3 cl', 'La Fée Parisienne absinthe'], ['7.5...",SHAKE all ingredients with ice and fine strain...,"Begin by gathering a coupe glass, your La Fée ..."
4,Absinthe Frappé,Old-fashioned glass,Mint sprig,"[['4.5 cl', 'La Fée Parisienne absinthe'], ['1...",SHAKE all ingredients with ice and fine strain...,Begin by chilling an old-fashioned glass and g...


In [8]:
[ing[1] for ing in eval(df.iloc[0]["ingredients"])]

['Pineapple (fresh)',
 'Havana Club 3 Year Old rum',
 'Lime juice (freshly squeezed)',
 'White caster sugar']

In [11]:
import ast
from collections import Counter

# Parse the ingredients column from string to actual lists
df['ingredients_list'] = df['ingredients'].apply(lambda x: [ing[1] for ing in eval(x)])

# Flatten all ingredients and count them
all_ingredients = []
for ingredients in df['ingredients_list']:
    all_ingredients.extend(ingredients)

ingredient_counts = Counter(all_ingredients)

# Get the 100 most common ingredients
top_100_ingredients = set([ing for ing, count in ingredient_counts.most_common(100)])
print(f"Top 100 ingredients: {len(top_100_ingredients)}")
print(ingredient_counts.most_common(200))  # Show top 20 for preview

Top 100 ingredients: 100
[('Lemon juice (freshly squeezed)', 1697), ('Lime juice (freshly squeezed)', 1627), ('Sugar syrup (65.0°brix, 2 sugar to 1 water rich syrup)', 1427), ('Rutte Dry Gin', 847), ('Angostura Aromatic Bitters', 621), ('Ketel One Vodka', 584), ('Bacardi Carta Blanca light rum', 507), ('Orange juice (freshly squeezed)', 491), ('Rémy Martin 1738 Cognac', 471), ('Pineapple juice (fresh pressed)', 440), ('Martini Extra Dry vermouth', 433), ('Bourbon whiskey', 432), ('Martini Rosso sweet vermouth', 407), ('Orange Bitters by Angostura', 382), ('De Kuyper Triple Sec (40%)', 379), ('Pasteurised egg white', 345), ('Luxardo Maraschino liqueur', 272), ('Apple juice', 267), ('Grapefruit juice (pink)', 261), ('Gin', 261), ('Thomas Henry Soda Water', 256), ('Giffard Grenadine syrup', 251), ('La Fée Parisienne absinthe', 240), ('St-Germain elderflower liqueur', 239), ('Cranberry juice (sweetened)', 236), ('Patrón Reposado tequila', 232), ('Italian red bitter liqueur', 232), ('Single

In [12]:
# Filter recipes that only contain ingredients from the top 100
def all_ingredients_in_top_100(ingredients_list):
    return all(ing in top_100_ingredients for ing in ingredients_list)

df_filtered = df[df['ingredients_list'].apply(all_ingredients_in_top_100)].copy()

print(f"Original recipes: {len(df)}")
print(f"Filtered recipes (only top 100 ingredients): {len(df_filtered)}")
print(f"\nFiltered dataset preview:")
df_filtered.head()

Original recipes: 6956
Filtered recipes (only top 100 ingredients): 1653

Filtered dataset preview:


,title,glass,garnish,ingredients,recipe,synthetic_instructions,ingredients_list
1,Abbey,Coupe glass,Orange zest twist,"[['4.5 cl', 'Rutte Dry Gin'], ['2.25 cl', 'Lil...",SHAKE all ingredients with ice and fine strain...,"Begin by gathering the Abbey, a bright, citrus...","[Rutte Dry Gin, Lillet Blanc (or other aromati..."
2,A.B.C. Cocktail,Nick & Nora glass,Lemon zest twist & Luxardo Maraschino cherry,"[['7 fresh', 'Mint leaves'], ['3 cl', 'Tawny p...",TEAR mint and place in shaker. Add other ingre...,Tear the fresh mint leaves and drop them into ...,"[Mint leaves, Tawny port, Rémy Martin 1738 Cog..."
3,Absinthe Cocktail,Coupe glass,Mint leaf,"[['3 cl', 'La Fée Parisienne absinthe'], ['7.5...",SHAKE all ingredients with ice and fine strain...,"Begin by gathering a coupe glass, your La Fée ...","[La Fée Parisienne absinthe, Chilled water, Su..."
8,Absolutely Fabulous,Flute glass,Strawberry,"[['3 cl', 'Ketel One Vodka'], ['4.5 cl', 'Cran...",SHAKE first two ingredients with ice and strai...,"Fill a flute with a ready chill, then add Kete...","[Ketel One Vodka, Cranberry juice (sweetened),..."
9,Acapulco,Collins glass,Pineapple wedge,"[['3 cl', 'Patrón Reposado tequila'], ['3 cl',...",SHAKE all ingredients with ice and strain into...,Fill a Collins glass with ice and set it aside...,"[Patrón Reposado tequila, Havana Club 3 Year O..."
